# Reencontra — Pipeline com dados reais de São Paulo

Fontes: API de municípios de SP do IBGE e estatísticas de desaparecimentos/localizações do MJSP, filtradas para UF SP.
O Airflow baixa as fontes em paralelo e confere os arquivos. O upload para o Volume é manual.

Execute as células na ordem. Os JSONs devem estar em `/Volumes/workspace/reencontra/chegada/municipios_ibge` e `desaparecimentos_sp`.
Bronze: COPY INTO. Prata: validação e MERGE pela chave. Ouro: fato estadual mensal e dimensões UF, mês e perfil.

As estatísticas do MJSP são agregadas por estado. Não são distribuídas entre os municípios.
Cada linha representa um indicador por período e perfil, não uma pessoa.
Quantidade ausente não significa zero: fica preservada na Bronze e na tabela de rejeitados.
O preenchimento dos campos não garante cobertura integral das ocorrências reais.

Quando há versões conflitantes da mesma chave, a Prata interrompe a execução para revisão, sem escolher um valor arbitrariamente.
Na amostra conferida: 1.080 combinações na Bronze, 1.079 aprovadas e uma rejeitada por quantidade ausente.

Pendências da entrega: segundo lote da nova fonte em outro dia de coleta, demonstração incremental desse lote,
consulta de impacto com medição do MVP, tela consumindo a Ouro, documentação e publicação do repositório.
Esta cópia remove a operação de atendimentos de demonstração. Não apaga tabelas nem arquivos do ambiente.


In [0]:
# Criar a area do projeto
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.reencontra")

# Criar o Volume para os arquivos
spark.sql("""
    CREATE VOLUME IF NOT EXISTS workspace.reencontra.chegada
""")

# Criar uma pasta para cada fonte
base = "/Volumes/workspace/reencontra/chegada"

dbutils.fs.mkdirs(f"{base}/municipios_ibge")
dbutils.fs.mkdirs(f"{base}/desaparecimentos_sp")

# Mostrar as pastas criadas
display(dbutils.fs.ls(base))


path,name,size,modificationTime
dbfs:/Volumes/workspace/reencontra/chegada/atendimentos_demo/,atendimentos_demo/,0,1794198246862
dbfs:/Volumes/workspace/reencontra/chegada/desaparecimentos_sp/,desaparecimentos_sp/,0,1794198246862
dbfs:/Volumes/workspace/reencontra/chegada/municipios_ibge/,municipios_ibge/,0,1794198246862
dbfs:/Volumes/workspace/reencontra/chegada/saida_mvp/,saida_mvp/,0,1794198246862


In [0]:
base = "/Volumes/workspace/reencontra/chegada"

for pasta in ["municipios_ibge", "desaparecimentos_sp"]:
    print(f"\nPasta: {pasta}")
    for arquivo in dbutils.fs.ls(f"{base}/{pasta}"):
        print(arquivo.name, "|", arquivo.size, "bytes")



Pasta: municipios_ibge
municipios_ibge_2026-11-07_ad608e896d8734da.json | 132275 bytes
municipios_ibge_2026-11-08_ed16b9044267dcba.json | 132275 bytes

Pasta: desaparecimentos_sp
desaparecimentos_sp_2026-11-09_795025704a6c63cc.json | 552861 bytes


In [0]:
for tabela in ["bronze_municipios", "bronze_desaparecimentos_sp"]:
    total = spark.table(f"workspace.reencontra.{tabela}").count()
    print(f"{tabela}: {total} linhas")

bronze_municipios: 1290 linhas
bronze_desaparecimentos_sp: 1080 linhas


In [0]:
# Criar a tabela Bronze dos municipios
spark.sql("""
    CREATE TABLE IF NOT EXISTS
    workspace.reencontra.bronze_municipios (
        municipio_id BIGINT,
        municipio_nome STRING,
        uf STRING,
        fonte STRING,
        url_fonte STRING
    )
    USING DELTA
""")

antes_ibge = spark.table("workspace.reencontra.bronze_municipios").count()

# Carregar somente arquivos ainda nao processados
resultado = spark.sql("""
    COPY INTO workspace.reencontra.bronze_municipios
    FROM '/Volumes/workspace/reencontra/chegada/municipios_ibge'
    FILEFORMAT = JSON
    FORMAT_OPTIONS ('multiLine' = 'true')
""")

display(resultado)

# Conferir a quantidade carregada
display(spark.sql("""
    SELECT COUNT(*) AS total_municipios
    FROM workspace.reencontra.bronze_municipios
"""))
depois_ibge = spark.table("workspace.reencontra.bronze_municipios").count()
print("IBGE — antes:", antes_ibge, "depois:", depois_ibge, "novas linhas:", depois_ibge - antes_ibge)


num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
0,0,0


total_municipios
1290


IBGE — antes: 1290 depois: 1290 novas linhas: 0


In [0]:
# Preparar os municipios e remover repeticoes identicas
spark.sql("""
    CREATE OR REPLACE TEMP VIEW municipios_preparados AS
    SELECT DISTINCT
        municipio_id,
        TRIM(municipio_nome) AS municipio_nome,
        UPPER(TRIM(uf)) AS uf
    FROM workspace.reencontra.bronze_municipios
    WHERE municipio_id IS NOT NULL
      AND municipio_nome IS NOT NULL
      AND TRIM(municipio_nome) <> ''
      AND UPPER(TRIM(uf)) = 'SP'
""")

# Conferir se existe mais de uma versao por municipio
conflitos = spark.sql("""
    SELECT municipio_id
    FROM municipios_preparados
    GROUP BY municipio_id
    HAVING COUNT(*) > 1
""").count()

assert conflitos == 0, "Ha versoes conflitantes para o mesmo municipio."

# Criar a tabela Prata
spark.sql("""
    CREATE TABLE IF NOT EXISTS
    workspace.reencontra.prata_municipios
    USING DELTA
    AS SELECT *
    FROM municipios_preparados
    WHERE 1 = 0
""")

# Inserir ou atualizar pela chave do municipio
spark.sql("""
    MERGE INTO workspace.reencontra.prata_municipios AS destino
    USING municipios_preparados AS origem
    ON destino.municipio_id = origem.municipio_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

# Conferir o total
display(spark.sql("""
    SELECT COUNT(*) AS total_municipios_prata
    FROM workspace.reencontra.prata_municipios
"""))


total_municipios_prata
645


In [0]:
tabela = "workspace.reencontra.bronze_desaparecimentos_sp"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tabela} (
        ano BIGINT,
        uf STRING,
        mes BIGINT,
        indicador STRING,
        faixa_etaria STRING,
        sexo STRING,
        quantidade_original STRING,
        campo_origem STRING,
        fonte STRING,
        url_fonte STRING,
        dado_sintetico BOOLEAN
    )
    USING DELTA
""")

antes = spark.table(tabela).count()

display(spark.sql(f"""
    COPY INTO {tabela}
    FROM '/Volumes/workspace/reencontra/chegada/desaparecimentos_sp'
    FILEFORMAT = JSON
    FORMAT_OPTIONS ('multiLine' = 'true')
"""))

depois = spark.table(tabela).count()

print("Registros antes:", antes)
print("Registros depois:", depois)
print("Novas linhas:", depois - antes)


num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
0,0,0


Registros antes: 2160
Registros depois: 2160
Novas linhas: 0


In [0]:
# Verificar que repetir os mesmos arquivos nao acrescenta linhas
for tabela, pasta in [
    ("bronze_municipios", "municipios_ibge"),
    ("bronze_desaparecimentos_sp", "desaparecimentos_sp"),
]:
    destino = f"workspace.reencontra.{tabela}"
    antes = spark.table(destino).count()
    spark.sql(f"""
        COPY INTO {destino}
        FROM '/Volumes/workspace/reencontra/chegada/{pasta}'
        FILEFORMAT = JSON
        FORMAT_OPTIONS ('multiLine' = 'true')
    """).collect()
    depois = spark.table(destino).count()
    assert antes == depois, f"A repeticao alterou a tabela {tabela}."
    print(tabela, "antes:", antes, "depois:", depois, "novas linhas:", depois - antes)


bronze_municipios antes: 1290 depois: 1290 novas linhas: 0
bronze_desaparecimentos_sp antes: 1080 depois: 1080 novas linhas: 0


In [0]:
spark.sql("""
    CREATE OR REPLACE TEMP VIEW desaparecimentos_sp_validados AS
    WITH preparados AS (
        SELECT *,
            TRY_CAST(quantidade_original AS BIGINT) AS quantidade
        FROM workspace.reencontra.bronze_desaparecimentos_sp
    )
    SELECT *,
        COALESCE(uf = 'SP', FALSE) AS uf_valida,

        COALESCE(
            ano BETWEEN 2023 AND YEAR(CURRENT_DATE())
            AND mes BETWEEN 1 AND 12,
            FALSE
        ) AS periodo_valido,

        COALESCE(
            indicador IN ('desaparecimento', 'localizacao')
            AND faixa_etaria IN (
                'crianca', 'adolescente', 'adulto',
                'idoso', 'nao_informada'
            )
            AND sexo IN (
                'masculino', 'feminino', 'nao_informado'
            ),
            FALSE
        ) AS perfil_valido,

        COALESCE(
            quantidade_original RLIKE '^[0-9]+$'
            AND quantidade IS NOT NULL
            AND quantidade >= 0,
            FALSE
        ) AS quantidade_valida

    FROM preparados
""")

display(spark.sql("""
    SELECT 'UF deve ser SP' AS regra,
           COUNT_IF(NOT uf_valida) AS reprovados
    FROM desaparecimentos_sp_validados

    UNION ALL

    SELECT 'Ano e mes validos',
           COUNT_IF(NOT periodo_valido)
    FROM desaparecimentos_sp_validados

    UNION ALL

    SELECT 'Indicador e perfil permitidos',
           COUNT_IF(NOT perfil_valido)
    FROM desaparecimentos_sp_validados

    UNION ALL

    SELECT 'Quantidade inteira e nao negativa',
           COUNT_IF(NOT quantidade_valida)
    FROM desaparecimentos_sp_validados
"""))


regra,reprovados
UF deve ser SP,0
Ano e mes validos,0
Indicador e perfil permitidos,0
Quantidade inteira e nao negativa,2


In [0]:
# Preservar os registros rejeitados para consulta
spark.sql("""
    CREATE OR REPLACE TABLE
    workspace.reencontra.rejeitados_desaparecimentos_sp
    USING DELTA
    AS
    SELECT *
    FROM desaparecimentos_sp_validados
    WHERE NOT (
        uf_valida AND periodo_valido
        AND perfil_valido AND quantidade_valida
    )
""")

# Remover somente repeticoes identicas dos aprovados
spark.sql("""
    CREATE OR REPLACE TEMP VIEW desaparecimentos_sp_aprovados AS
    SELECT DISTINCT
        uf, ano, mes, indicador, faixa_etaria, sexo,
        quantidade, campo_origem, fonte, url_fonte
    FROM desaparecimentos_sp_validados
    WHERE uf_valida AND periodo_valido
      AND perfil_valido AND quantidade_valida
""")

# Impedir que duas versoes conflitantes usem a mesma chave
conflitos = spark.sql("""
    SELECT uf, ano, mes, indicador, faixa_etaria, sexo
    FROM desaparecimentos_sp_aprovados
    GROUP BY uf, ano, mes, indicador, faixa_etaria, sexo
    HAVING COUNT(*) > 1
""").count()

assert conflitos == 0, "Existem versoes conflitantes para a mesma chave."

spark.sql("""
    CREATE TABLE IF NOT EXISTS
    workspace.reencontra.prata_desaparecimentos_sp
    USING DELTA
    AS SELECT *
    FROM desaparecimentos_sp_aprovados
    WHERE 1 = 0
""")

spark.sql("""
    MERGE INTO workspace.reencontra.prata_desaparecimentos_sp d
    USING desaparecimentos_sp_aprovados o
    ON  d.uf = o.uf
    AND d.ano = o.ano
    AND d.mes = o.mes
    AND d.indicador = o.indicador
    AND d.faixa_etaria = o.faixa_etaria
    AND d.sexo = o.sexo
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

display(spark.sql("""
    SELECT 'Prata: aprovados' AS destino, COUNT(*) AS linhas
    FROM workspace.reencontra.prata_desaparecimentos_sp

    UNION ALL

    SELECT 'Rejeitados', COUNT(*)
    FROM workspace.reencontra.rejeitados_desaparecimentos_sp
"""))


destino,linhas
Prata: aprovados,1079
Rejeitados,2


In [0]:
# Dimensao estado, baseada nos municipios do IBGE
spark.sql("""
    CREATE OR REPLACE TABLE workspace.reencontra.ouro_dim_uf
    USING DELTA AS
    SELECT
        uf,
        COUNT(DISTINCT municipio_id) AS quantidade_municipios
    FROM workspace.reencontra.prata_municipios
    GROUP BY uf
""")

# Dimensao de periodo mensal
spark.sql("""
    CREATE OR REPLACE TABLE
    workspace.reencontra.ouro_dim_mes_desaparecimentos
    USING DELTA AS
    SELECT DISTINCT
        CAST(ano * 100 + mes AS INT) AS mes_id,
        ano,
        mes,
        MAKE_DATE(CAST(ano AS INT), CAST(mes AS INT), 1)
            AS inicio_mes
    FROM workspace.reencontra.prata_desaparecimentos_sp
""")

# Dimensao de perfil agregado
spark.sql("""
    CREATE OR REPLACE TABLE
    workspace.reencontra.ouro_dim_perfil_desaparecimentos
    USING DELTA AS
    SELECT DISTINCT
        SHA2(CONCAT_WS('|', faixa_etaria, sexo), 256) AS perfil_id,
        faixa_etaria,
        sexo
    FROM workspace.reencontra.prata_desaparecimentos_sp
""")

# Fato: quantidade por estado, mes, perfil e indicador
spark.sql("""
    CREATE OR REPLACE TABLE
    workspace.reencontra.ouro_fato_desaparecimentos
    USING DELTA AS
    SELECT
        uf,
        CAST(ano * 100 + mes AS INT) AS mes_id,
        SHA2(CONCAT_WS('|', faixa_etaria, sexo), 256) AS perfil_id,
        indicador,
        quantidade,
        fonte,
        url_fonte
    FROM workspace.reencontra.prata_desaparecimentos_sp
""")

display(spark.sql("""
    SELECT 'Fato' AS tabela, COUNT(*) AS linhas
    FROM workspace.reencontra.ouro_fato_desaparecimentos

    UNION ALL

    SELECT 'Dimensao estado', COUNT(*)
    FROM workspace.reencontra.ouro_dim_uf

    UNION ALL

    SELECT 'Dimensao mes', COUNT(*)
    FROM workspace.reencontra.ouro_dim_mes_desaparecimentos

    UNION ALL

    SELECT 'Dimensao perfil', COUNT(*)
    FROM workspace.reencontra.ouro_dim_perfil_desaparecimentos
"""))


tabela,linhas
Fato,1079
Dimensao estado,1
Dimensao mes,36
Dimensao perfil,15


In [0]:
# Conferir a ligacao da fato com as tres dimensoes
resultado = spark.sql("""
    SELECT f.uf, m.ano, m.mes, p.faixa_etaria, p.sexo,
           f.indicador, f.quantidade, u.quantidade_municipios
    FROM workspace.reencontra.ouro_fato_desaparecimentos f
    JOIN workspace.reencontra.ouro_dim_uf u ON f.uf = u.uf
    JOIN workspace.reencontra.ouro_dim_mes_desaparecimentos m
        ON f.mes_id = m.mes_id
    JOIN workspace.reencontra.ouro_dim_perfil_desaparecimentos p
        ON f.perfil_id = p.perfil_id
""")
assert resultado.count() == spark.table(
    "workspace.reencontra.ouro_fato_desaparecimentos"
).count(), "As dimensoes perderam ou multiplicaram linhas da fato."
display(resultado.orderBy("ano", "mes", "indicador", "faixa_etaria", "sexo"))


uf,ano,mes,faixa_etaria,sexo,indicador,quantidade,quantidade_municipios
SP,2023,1,adolescente,feminino,desaparecimento,171,645
SP,2023,1,adolescente,masculino,desaparecimento,92,645
SP,2023,1,adolescente,nao_informado,desaparecimento,0,645
SP,2023,1,adulto,feminino,desaparecimento,233,645
SP,2023,1,adulto,masculino,desaparecimento,798,645
SP,2023,1,adulto,nao_informado,desaparecimento,1,645
SP,2023,1,crianca,feminino,desaparecimento,18,645
SP,2023,1,crianca,masculino,desaparecimento,14,645
SP,2023,1,crianca,nao_informado,desaparecimento,0,645
SP,2023,1,idoso,feminino,desaparecimento,34,645
